Multiple tests. A patient tests positive twice on independent tests (both 99% accurate, disease prevalence 1 in 10,000). What is P(sick) after both tests? Use the posterior from the first test as the prior for the second.

#### Probability of Disease After Two Positive Tests

##### Problem

A patient tests positive twice on independent tests. Each test is 99% accurate, and the disease prevalence is 1 in 10,000. What is the probability the patient is sick after both tests? Use the posterior from the first test as the prior for the second.

---

##### Step 1: Define Events and Given Data

- **Disease prevalence**: \( P(D) = \frac{1}{10{,}000} = 0.0001 \)
- **Test accuracy**: 99% for both sensitivity and specificity
  - Sensitivity: \( P(+ \mid D) = 0.99 \)
  - Specificity: \( P(- \mid \text{no } D) = 0.99 \)
  - False positive rate: \( P(+ \mid \text{no } D) = 0.01 \)

---

##### Step 2: First Test — Apply Bayes' Theorem

Prior odds before first test:

\[
\text{Odds}(D) = \frac{P(D)}{P(\text{no } D)} = \frac{0.0001}{0.9999}
\]

Likelihood ratio for a positive test:

\[
LR_+ = \frac{P(+ \mid D)}{P(+ \mid \text{no } D)} = \frac{0.99}{0.01} = 99
\]

Posterior odds after first positive test:

\[
\text{Odds}(D \mid +) = \frac{0.0001}{0.9999} \times 99 \approx 0.00010001 \times 99 \approx 0.009901
\]

Convert to probability:

\[
P(D \mid +) = \frac{\text{Odds}}{1 + \text{Odds}} \approx \frac{0.009901}{1.009901} \approx 0.009803
\]

So after the first positive test, probability of disease ≈ **0.9803%**.

---

##### Step 3: Second Test — Use Posterior from First as Prior

Now prior odds for the second test = 0.009901 (from Step 2).

Apply the same likelihood ratio \( 99 \):

\[
\text{Odds}(D \mid ++) = 0.009901 \times 99 \approx 0.9802
\]

Convert to probability:

\[
P(D \mid ++) = \frac{0.9802}{1 + 0.9802} \approx \frac{0.9802}{1.9802} \approx 0.495
\]

---

##### Step 4: Conclusion

After two independent positive tests, the probability the patient is sick ≈ **49.5%**.

Even with two highly accurate tests, low disease prevalence means the chance is still about a coin flip.

\[
\boxed{0.495}
\]

Smoothing impact. Run the spam classifier with smoothing values of 0.01, 0.1, 1.0, and 10.0. How do the top word probabilities change? What happens with smoothing=0 and a word that appears only in ham?

In [2]:
"""
Smoothing Impact on Naive Bayes Spam Classifier
Reproduces all tables and the smoothing=0 demonstration.
"""

# ---------------------------
# Training data
# ---------------------------
vocab = ["free", "meeting", "viagra", "project", "lunch"]
ham_counts  = {"free": 1,  "meeting": 20, "viagra": 0,  "project": 30, "lunch": 25}
spam_counts = {"free": 20, "meeting": 0,  "viagra": 25, "project": 0,  "lunch": 5}

ham_total  = sum(ham_counts.values())   # 100
spam_total = sum(spam_counts.values())  # 100
V = len(vocab)

# ---------------------------
# Helper: smoothed probability
# ---------------------------
def prob(word, counts, total, alpha):
    """P(word | class) with Laplace smoothing."""
    return (counts[word] + alpha) / (total + alpha * V)

# ---------------------------
# Print probability tables
# ---------------------------
def print_table(alpha):
    denom_ham  = ham_total  + alpha * V
    denom_spam = spam_total + alpha * V
    print(f"\n=== alpha = {alpha} ===")
    print(f"Ham denominator:  {denom_ham}")
    print(f"Spam denominator: {denom_spam}")
    print(f"{'Word':<10}{'P(w|ham)':<15}{'P(w|spam)':<15}")
    print("-" * 40)
    for w in vocab:
        ph = prob(w, ham_counts,  ham_total,  alpha)
        ps = prob(w, spam_counts, spam_total, alpha)
        print(f"{w:<10}{ph:<15.4f}{ps:<15.4f}")

for alpha in [0.01, 0.1, 1.0, 10.0]:
    print_table(alpha)

# ---------------------------
# Demonstration: smoothing = 0
# ---------------------------
print("\n=== alpha = 0 : zero-probability trap ===")

email = ["free", "viagra", "meeting"]

# Unsmoothed likelihoods for spam
spam_likelihood = 1.0
ham_likelihood  = 1.0

for w in email:
    p_spam = prob(w, spam_counts, spam_total, 0.0)
    p_ham  = prob(w, ham_counts,  ham_total,  0.0)
    spam_likelihood *= p_spam
    ham_likelihood  *= p_ham
    print(f"P({w}|spam) = {p_spam:.4f}   P({w}|ham) = {p_ham:.4f}")

print(f"\nP(email|spam) = {spam_likelihood}")
print(f"P(email|ham)  = {ham_likelihood}")

if spam_likelihood == 0:
    print("\n>>> Spam likelihood collapsed to ZERO.")
    print(">>> The word 'meeting' (never seen in spam) vetoed the spam class.")


=== alpha = 0.01 ===
Ham denominator:  76.05
Spam denominator: 50.05
Word      P(w|ham)       P(w|spam)      
----------------------------------------
free      0.0133         0.3998         
meeting   0.2631         0.0002         
viagra    0.0001         0.4997         
project   0.3946         0.0002         
lunch     0.3289         0.1001         

=== alpha = 0.1 ===
Ham denominator:  76.5
Spam denominator: 50.5
Word      P(w|ham)       P(w|spam)      
----------------------------------------
free      0.0144         0.3980         
meeting   0.2627         0.0020         
viagra    0.0013         0.4970         
project   0.3935         0.0020         
lunch     0.3281         0.1010         

=== alpha = 1.0 ===
Ham denominator:  81.0
Spam denominator: 55.0
Word      P(w|ham)       P(w|spam)      
----------------------------------------
free      0.0247         0.3818         
meeting   0.2593         0.0182         
viagra    0.0123         0.4727         
project   0.3827 

Add features. Extend the NaiveBayes class to also use message length (short/long) as a feature alongside word counts. Estimate P(short|spam) and P(short|ham) from the training data and fold it into the prediction score.

In [3]:
import math
from collections import defaultdict


class NaiveBayes:
    """
    Naive Bayes classifier with:
      - word-count features (multinomial)
      - message length feature: 'short' or 'long' (binary)
    """

    def __init__(self, smoothing=1.0, length_threshold=5):
        self.smoothing = smoothing
        self.length_threshold = length_threshold  # words <= threshold => 'short'

        # Word feature statistics
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

        # Length feature statistics: P(short | class)
        self.length_counts = defaultdict(lambda: defaultdict(int))  # length_counts[cls]['short'] = n

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _length_bucket(self, document):
        """Return 'short' or 'long' based on word count."""
        n_words = len(document.lower().split())
        return "short" if n_words <= self.length_threshold else "long"

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    def train(self, documents, labels):
        for doc, label in zip(documents, labels):
            self.class_counts[label] += 1

            # ---- word counts ----
            words = doc.lower().split()
            for word in words:
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

            # ---- length feature ----
            bucket = self._length_bucket(doc)
            self.length_counts[label][bucket] += 1

    # ------------------------------------------------------------------
    # Probability estimates
    # ------------------------------------------------------------------
    def _p_word_given_class(self, word, cls):
        count = self.word_counts[cls].get(word, 0)
        total = self.class_word_totals[cls]
        V = len(self.vocab)
        return (count + self.smoothing) / (total + self.smoothing * V)

    def _p_length_given_class(self, bucket, cls):
        """
        P(bucket | class) with Laplace smoothing over 2 buckets
        (short / long) to avoid zero probabilities.
        """
        count = self.length_counts[cls].get(bucket, 0)
        total = self.class_counts[cls]
        return (count + self.smoothing) / (total + self.smoothing * 2)

    def p_short_given_spam(self):
        return self._p_length_given_class("short", "spam")

    def p_short_given_ham(self):
        return self._p_length_given_class("short", "ham")

    # ------------------------------------------------------------------
    # Prediction
    # ------------------------------------------------------------------
    def predict(self, document):
        words = document.lower().split()
        total_docs = sum(self.class_counts.values())
        V = len(self.vocab)
        bucket = self._length_bucket(document)

        best_class = None
        best_score = float("-inf")

        for cls in self.class_counts:
            # Prior
            score = math.log(self.class_counts[cls] / total_docs)

            # Word likelihoods
            for word in words:
                score += math.log(self._p_word_given_class(word, cls))

            # Length likelihood
            score += math.log(self._p_length_given_class(bucket, cls))

            if score > best_score:
                best_score = score
                best_class = cls

        return best_class

    # ------------------------------------------------------------------
    # Debug helper
    # ------------------------------------------------------------------
    def report_length_probs(self):
        print("Length feature probabilities (with smoothing = %.2f):" % self.smoothing)
        for cls in self.class_counts:
            p_short = self._p_length_given_class("short", cls)
            p_long = self._p_length_given_class("long", cls)
            print(f"  {cls:<5} -> P(short|{cls}) = {p_short:.4f}   "
                  f"P(long|{cls}) = {p_long:.4f}")


# ======================================================================
# Demo
# ======================================================================
if __name__ == "__main__":
    # Training corpus: (message, label)
    train_data = [
        # -------- spam --------
        ("free viagra now", "spam"),
        ("win free money now", "spam"),
        ("cheap viagra deal", "spam"),
        ("free cash prize", "spam"),
        ("viagra discount now", "spam"),
        ("limited time free offer", "spam"),
        ("click here for free money", "spam"),
        ("hot deal viagra cheap", "spam"),

        # -------- ham --------
        ("project meeting tomorrow", "ham"),
        ("lunch meeting at noon", "ham"),
        ("can we reschedule the project meeting", "ham"),
        ("please review the project document", "ham"),
        ("team lunch this friday", "ham"),
        ("the project deadline is next week", "ham"),
        ("meeting notes from yesterday", "ham"),
        ("let us discuss the project plan over lunch", "ham"),
    ]

    nb = NaiveBayes(smoothing=1.0, length_threshold=5)
    nb.train([d for d, _ in train_data], [l for _, l in train_data])

    # ---- Report P(short | class) ----
    nb.report_length_probs()

    print("\nDirect estimates:")
    print(f"  P(short | spam) = {nb.p_short_given_spam():.4f}")
    print(f"  P(short | ham)  = {nb.p_short_given_ham():.4f}")

    # ---- Predictions ----
    tests = [
        "free viagra",                         # short spammy
        "project meeting tomorrow",            # short hammy (actually medium)
        "click here for free money now",       # short spammy
        "can we reschedule the project meeting tomorrow afternoon",  # long hammy
        "win free cash prize now",             # short spammy
    ]

    print("\nPredictions:")
    for msg in tests:
        bucket = nb._length_bucket(msg)
        label = nb.predict(msg)
        print(f"  [{bucket:<5}] {msg!r:<60} -> {label}")

Length feature probabilities (with smoothing = 1.00):
  spam  -> P(short|spam) = 0.9000   P(long|spam) = 0.1000
  ham   -> P(short|ham) = 0.6000   P(long|ham) = 0.4000

Direct estimates:
  P(short | spam) = 0.9000
  P(short | ham)  = 0.6000

Predictions:
  [short] 'free viagra'                                                -> spam
  [short] 'project meeting tomorrow'                                   -> ham
  [long ] 'click here for free money now'                              -> spam
  [long ] 'can we reschedule the project meeting tomorrow afternoon'   -> ham
  [short] 'win free cash prize now'                                    -> spam
